In [1]:
import os
import glob
import numpy as np
import pandas as pd
import itertools

import utils
import json

from scipy import stats
from statsmodels.stats.multitest import multipletests
from goatools.utils import read_geneset
from goatools.obo_parser import GODag
from goatools.obo_parser import OBOReader
from goatools.anno.idtogos_reader import IdToGosReader
from goatools.go_enrichment import GOEnrichmentStudy
from goatools.base import download_go_basic_obo

In [2]:
fs = 12
src = os.pardir + os.sep + 'raw' + os.sep
dst = os.pardir + os.sep + 'outputs' + os.sep
if not os.path.isdir(dst):
    os.mkdir(dst)

# List the name of the lines, their repetitions
# and pairs to compare
reps = [1,2]
lines = ['BA24','BM24','IA24','IM24','ZA24','ZM24']
full_lines = list(itertools.chain(*[[ '{}_{}'.format(l,r) for r in reps ] for l in lines]))
comparisons = [(0,1),(2,3),(4,5),(0,2),(0,4),(1,3),(1,5)]

# Read the count number file
filename = src + 'readCounts_Xa7_msu_v7.csv'
data = pd.read_csv(filename).set_index('Geneid')
genelength = data['Length']
data = data[full_lines] + 1
rpk = data.div(genelength, axis='index')
print('Original dataframe dimensions:\t', rpk.shape)

# Standardize expression levels
# Remove genes where at least one population reports all zero values
min_tpm = 100
tpm = utils.count_normalization(data, rpk, lines, reps, normalize_flag='DESeq2')
tpm = tpm.loc[tpm.max(axis = 1) > min_tpm]
print('After initial cleanup:\t', tpm.shape)

Original dataframe dimensions:	 (55986, 12)
After initial cleanup:	 (19601, 12)


In [3]:
filename = os.pardir + os.sep + 'raw' + os.sep + 'rice_GO_terms.csv'
if not os.path.isfile(filename):
    gocols = ['Locus', 'GO ID', 'Type', 'GO term', 'TAIRLocus']
    goslimfile = os.pardir + os.sep + 'raw' + os.sep + 'osa1_r7.all_models.GOSlim.txt'
    goslim = pd.read_csv(goslimfile, sep='\t', header=None).drop(columns=4)
    goslim.columns = gocols
    goslim['Locus'] = goslim['Locus'].str.replace('mRNA','gene')
    foo = ['' for _ in range(len(goslim)) ]
    for i in range(len(foo)):
        if 'LOC' in goslim.iloc[i,0]:
            foo[i] = goslim.iloc[i,0].split('.')[0]
        else:
            foo[i] = goslim.iloc[i,0]
    goslim['Locus'] = foo
    print(goslim.shape)
    goslim['TAIRLocus'] = goslim['TAIRLocus'].str.replace('TAIR:','')
    goterms = goslim.drop_duplicates(ignore_index=True)
    goterms.to_csv(filename, index=False)

dfterms = pd.read_csv(filename)
print(dfterms.shape)
dfterms

(132323, 5)


,Locus,GO ID,Type,GO term,TAIRLocus
0,LOC_Os01g01010,GO:0030234,F,enzyme regulator activity,AT3G59570
1,LOC_Os01g01010,GO:0007165,P,signal transduction,AT3G59570
2,LOC_Os01g01010,GO:0006139,P,"nucleobase, nucleoside, nucleotide and nucleic...",AT3G59570
3,LOC_Os01g01010,GO:0009056,P,catabolic process,AT3G59570
4,LOC_Os01g01010,GO:0005622,C,intracellular,AT3G59570
...,...,...,...,...,...
132318,ChrUn.fgenesh.gene.96,GO:0008152,P,metabolic process,AT1G76680
132319,ChrUn.fgenesh.gene.96,GO:0008150,P,biological_process,AT1G76680
132320,ChrUn.fgenesh.gene.96,GO:0009987,P,cellular process,AT1G76680
132321,ChrUn.fgenesh.gene.96,GO:0006629,P,lipid metabolic process,AT1G76680


In [5]:
# Load the go-basic.obo file if it exists, else download it.
fin_obo   = src + "go-basic.obo"
if not os.path.isfile(fin_obo):
    fin_obo = download_go_basic_obo()

with open(fin_obo, encoding="utf-8") as f:
    obo = f.read().replace('"','')

filename = src + 'clean_oryza_go_terms.csv'
if not os.path.isfile(filename):
    terms = obo.split('\n\n[Term]\n')[1:]
    terms[-1] = terms[-1].split('\n\n[Typedef]\n')[0]
    
    goterms = dict()
    for term in terms:
        jstr = '{' + ','.join(['"' + t.strip().replace('\\','').replace(': ','":"',1) + '"' for t in term.split('\n') ]) + '}'
        d = json.loads(jstr)
        goterms[ d['id'] ] = d
    
    goreplace = dict()
    for key in goterms:
        goreplace[key] = []
        if 'replaced_by' in goterms[key]:
            goreplace[key].append(goterms[key]['replaced_by'])
        if 'consider' in goterms[key]:
            goreplace[key].append(goterms[key]['consider'])
        if len(goreplace[key]) == 0:
            del goreplace[key]

    goremove = []
    for key in goterms:
        if ('is_obsolete' in goterms[key]) and (key not in goreplace):
            if goterms[key]['is_obsolete'] == 'true':
                goremove.append(key)
    print(len(goremove))
    
    foo = ','.join(dfterms['GO ID'].values)
    for term in goreplace:
        foo = foo.replace(term , goreplace[term][0])
    for term in goremove:
        foo = foo.replace(term, 'blank')
    dfterms['GO ID'] = foo.split(',')
    dfterms = dfterms[dfterms['GO ID'] != 'blank']
    dfterms.to_csv( filename , index = False)

dfterms = pd.read_csv(filename)
dfterms.head()

,Locus,GO ID,Type,GO term,TAIRLocus
0,LOC_Os01g01010,GO:0030234,F,enzyme regulator activity,AT3G59570
1,LOC_Os01g01010,GO:0007165,P,signal transduction,AT3G59570
2,LOC_Os01g01010,GO:0006139,P,"nucleobase, nucleoside, nucleotide and nucleic...",AT3G59570
3,LOC_Os01g01010,GO:0009056,P,catabolic process,AT3G59570
4,LOC_Os01g01010,GO:0005622,C,intracellular,AT3G59570


In [6]:
# List all unique TAIR loci associated with orthogroups and write to a file
tair_file = src + "all_os_loci.txt"
with open(tair_file, "w") as fp:
    fp.write("\n".join(dfterms["Locus"].unique()))

# Create TAIR Locus - GO ID associations and write to file
assoc_file = src + "os_loci_go_association.txt"
tair_grouped = dfterms.groupby("Locus")["GO ID"].aggregate(set)
with open(assoc_file, "w") as fp:
    for t in tair_grouped.index:
        fp.write(t + "\t" + ";".join(list(tair_grouped[t])) + "\n")

In [11]:
annoobj

In [8]:
# Populate necessary variables to run GO Enrichment
godag = GODag(fin_obo)
annoobj = IdToGosReader(assoc_file, godag=godag)
id2gos = annoobj.get_id2gos()

# GO Enrichment
goeaobj = GOEnrichmentStudy(
    population_ids,
    annoobj.get_id2gos(),
    godag,
    methods=['bonferroni', 'fdr_bh'],
    pvalcalc='fisher_scipy_stats')

results = goeaobj.run_study_nts(study_ids)

../raw/go-basic.obo: fmt(1.2) rel(2026-01-23) 42,036 Terms
HMS:0:00:00.429410 127,495 annotations READ: ../raw/os_loci_go_association.txt 
25463 IDs in loaded association branch, biological_process


NameError: name 'population_ids' is not defined

In [91]:
# Collect GO Enrichment results, format and write to csv file.
goea_out = src + "gotest.csv"

# Column headers
hdrs = ["Namespace", "GO ID", "Enriched/Purified", "GO Terms", "pval_uncorr", "Benjamimi/Hochberg", "Bonferroni", "Study Ratio", "Population Ratio"]

# Row values pattern
pat = "{NS}\t{GO}\t{e}\t{GOTERM}\t{PVAL:8.2e}\t\
       {BH:8.2e}\t{BONF:8.2e}\t{RS:>12}\t{RP:>12}\n"

# Write to file, one row at a time: iff adjusted p-value < 0.05
with open(goea_out, "w") as fp:
    fp.write("\t".join(hdrs) + "\n")
    for ntd in sorted(results, key=lambda nt: [nt.p_uncorrected, nt.GO]):
        if ntd.p_fdr_bh < 5:
            ogs = dfterms[dfterms["GO ID"] == ntd.GO]["Locus"].tolist()
            fp.write(pat.format(
                    NS=ntd.NS,
                    GO=ntd.GO,
                    e=ntd.enrichment,
                    GOTERM=ntd.goterm.name,
                    RS='{}/{}'.format(*ntd.ratio_in_study),
                    RP='{}/{}'.format(*ntd.ratio_in_pop),
                    PVAL=ntd.p_uncorrected,
                    BONF=ntd.p_bonferroni,
                    BH=ntd.p_fdr_bh))

print(f"Wrote results to {goea_out}")

Wrote results to ../raw/gotest.csv


In [1]:
animal_list = [
    ['Gray fox', 'Canidae'],
    ['Jaguar', 'Felidae'],
    ['Canadian Lynx', 'Felidae'],
    ['Cloud lepard', 'Felidae'],
    ['Bobcat', 'Felidae'],
    ['Fennec Fox', 'Canidae'],
    ['Ocelot', 'Felidae'],
    ['Domestic cat', 'Felidae'],
    ['Red Fox', 'Canidae'],
    ['Cheetah', 'Felidae'],
    ['Lion', 'Felidae'],
    ['Swift Fox', 'Canidae'],
    ['Eurasian lynx', 'Felidae']
]
catlike = []

# Loop through the big list of animals
for jj in range(len(animal_list)):
    # For each `animal_list` entry, check the second value (family name)
    if animal_list[jj][1] == 'Felidae':
        # If a feline, then append just the common name--the first value
        catlike.append(animal_list[jj][0])

print(catlike)

['Jaguar', 'Canadian Lynx', 'Cloud lepard', 'Bobcat', 'Ocelot', 'Domestic cat', 'Cheetah', 'Lion', 'Eurasian lynx']


What would be the way I can make the review code return the first name in the nested list, the name of the animal, instead of both values in the list, the name of the animal then the family name?